<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Computer-Networks/02-link-lan-and-access-networks.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Computer Networks guideline](Computer-Networks.html)


## **Link, LAN, and Access Networks**

Chapter 1 followed an IP packet across the Internet. This chapter slows down at the first step: **how does a host deliver that packet to one adjacent device?** Before a router can make an Internet-layer decision, a laptop must gain access to Ethernet or Wi-Fi, package the datagram inside a frame, contend for a finite medium, and identify the link-layer address of its next hop.

The running case is a laptop opening a remote HTTPS site through campus Wi-Fi. The destination server may be thousands of kilometres away, but the first transmitted frame is not addressed to that server's network interface. It is delivered locally through an access point and switches to the laptop's default gateway. That separation between **end-to-end IP destination** and **one-hop link destination** is the central idea of the chapter.

::: {.callout-note}
On a first reading, prioritize frames, MAC addresses, switch learning, ARP, and Wi-Fi contention. CRC arithmetic, ALOHA throughput, spanning-tree roles, and access-network variants add depth but are not required to understand the first-hop story.
:::

The link layer is where abstract packets meet physical constraints. Signals weaken, shared transmitters interfere, finite frames can be corrupted, and local loops can duplicate traffic indefinitely. Its mechanisms are therefore shaped by the properties of a particular link rather than by one universal end-to-end design.


### **The Role of the Link Layer**

#### **Nodes, Links, Frames, and Hops**

A **node** is any device participating in a link-layer exchange: a host, access point, switch, or router interface. A **link** is the communication channel between adjacent nodes. A **frame** is the bounded link-layer unit carried over that channel. One traversal from a node to an adjacent node is a **hop**.

The same IP datagram can cross several different link technologies. It might begin inside an 802.11 Wi-Fi frame, move through an Ethernet frame on a campus LAN, cross an ISP optical link using another framing scheme, and finally enter the server's Ethernet network. At each routed hop, the old frame is consumed and a new frame is created. The enclosed datagram survives logically end to end, subject to fields routers are allowed to update such as TTL or hop limit.

This gives each layer a different scope:

| Identifier or unit | Scope | Typical lifetime |
|---|---|---|
| Link-layer address and frame | One local link or bridged LAN | Replaced at a routed hop |
| IP address and datagram | Internetwork path | Usually end to end |
| Transport ports and segment/datagram | Endpoint processes | End to end |
| Application name and message | Application meaning | Defined by the application |

#### **Link-Layer Services and Implementation Location**

A link layer may provide several services, but no technology must provide all of them:

- **Framing** marks the beginning and end of a unit and identifies its payload type.
- **Medium access control** decides when a node may use a shared channel.
- **Link addressing** identifies interfaces or groups within a local domain.
- **Error detection** rejects frames whose received bits are inconsistent with a check value.
- **Error correction or local retransmission** repairs some errors without waiting for an end-to-end transport.
- **Flow control** prevents a fast sender from overwhelming an adjacent receiver.
- **Link management** establishes, monitors, secures, and tears down attachment.

Implementation is split across hardware and software. A Network Interface Controller (NIC) commonly handles signal encoding, frame transmission/reception, CRC checking, address filtering, queues, and direct memory access. A device driver manages descriptors and exposes the interface to the operating system. The operating system connects link delivery to ARP/NDP, IP, packet filtering, bridging, and sockets.

This division is performance-sensitive. At 10 Gb/s, minimum-sized Ethernet frames can arrive millions of times per second. Performing every bit-level action in a general-purpose application would consume excessive CPU and introduce unpredictable latency, so fast-path operations are implemented in NIC logic, kernel code, and specialized switch hardware.

#### **Point-to-Point and Shared-Medium Links**

A **point-to-point** link has one sender and one receiver at each end. There is no competition among several stations for the same channel, although queues and errors can still occur. Modern full-duplex switched Ethernet behaves this way on each host-to-switch cable.

A **shared medium** allows multiple stations to transmit into a common channel. Classic coaxial Ethernet, a hub-based Ethernet collision domain, and a Wi-Fi channel are examples. The stations need a multiple-access protocol because simultaneous transmissions can interfere or collide.

Do not confuse a physically shared medium with a logically shared service. Several switched Ethernet hosts share uplink capacity, but each access cable can still be full duplex and collision-free. Conversely, Wi-Fi stations may have different radio visibility even when they share the same access point, making contention more difficult than "everyone hears everyone."

**Checkpoint.** A router forwards an IP datagram between links; a switch extends a link-layer domain by forwarding frames; an access point bridges wireless stations into a distribution system while also managing radio access.


### **Physical Transmission Foundations**

#### **Copper, Fiber, and Wireless Media**

The link layer depends on the medium below it because that medium determines attenuation, interference, propagation, duplex behavior, and error patterns.

| Medium | Signal | Strengths | Typical constraints |
|---|---|---|---|
| Twisted-pair copper | Electrical voltage changes | Inexpensive, easy to terminate, can carry power | Attenuation, electromagnetic interference, distance limits |
| Coaxial cable | Electrical signal with shielding | Strong shielding and shared broadband use | Shared capacity and installation constraints |
| Optical fiber | Modulated light | High rates, long distance, low electromagnetic interference | Optical components, bending/loss, installation cost |
| Terrestrial radio | Electromagnetic waves through space | Mobility and no cable | Interference, obstacles, fading, shared spectrum |
| Satellite radio | Long-distance radio relay | Wide coverage and remote access | Propagation delay, weather/link budget, shared capacity |

Fiber does not mean that packets move instantaneously. Light travels more slowly in glass than in vacuum, and serialization, switching, queues, and protocol processing still apply. Wireless does not mean that every device receives the same signal quality: distance, walls, orientation, competing transmitters, and multipath propagation produce different conditions even within one room.

#### **Bit Rate, Bandwidth, Noise, and Attenuation**

**Bit rate** is the number of data bits transmitted per second. The word **bandwidth** is overloaded: networking discussions often use it informally for bit rate, while communications theory uses it for a frequency range measured in hertz. Keeping the unit visible prevents confusion.

A receiver must distinguish symbols despite noise and distortion. Signal-to-noise ratio (SNR) compares useful signal power with unwanted power. Shannon's capacity result gives a theoretical upper bound for an idealized noisy channel:

$$
C \leq B \log_2\left(1+\frac{S}{N}\right),
$$

where $C$ is capacity in bit/s, $B$ is channel bandwidth in hertz, and $S/N$ is the linear signal-to-noise ratio. The formula does not directly tell an Ethernet or Wi-Fi implementation which modulation to use. It explains a fundamental trade-off: higher rate requires more frequency resource, better SNR, more sophisticated coding, or some combination.

**Attenuation** is loss of signal power as distance increases. A positive loss value in decibels can be written as

$$
Loss_{dB}=10\log_{10}\left(\frac{P_{in}}{P_{out}}\right).
$$

Because decibels are logarithmic, losses along a path can be added. Repeaters, amplifiers, and regenerated digital links address different aspects of weakening, but they cannot reconstruct information that was never received reliably.

#### **Line Encoding, Modulation, and Multiplexing Overview**

Raw bits are abstract; the transmitter must map them to physical symbols.

- **Line encoding** maps bits to baseband signal patterns and may ensure enough transitions for clock recovery.
- **Modulation** changes a carrier's amplitude, phase, frequency, or a combination to encode symbols.
- A symbol can represent more than one bit, but distinguishing more symbol states generally needs better SNR.
- **Coding** adds controlled redundancy so a receiver can detect or correct errors.
- **Multiplexing** lets several users or channels share a physical resource.

Time-division multiplexing assigns time intervals; frequency-division and wavelength-division multiplexing assign frequency or optical bands; orthogonal frequency-division techniques divide data across many subcarriers; spatial multiplexing uses multiple antennas or paths. These mechanisms belong mainly to physical-layer and digital-communications study. At the networking level, their consequence is that the advertised PHY rate and the useful application goodput are not the same quantity.


### **Framing and Error Handling**

#### **Byte-Oriented and Bit-Oriented Framing**

A receiver observes a stream of symbols or bytes and needs to recover discrete frames. A framing scheme must identify boundaries without confusing payload data for control markers.

**Byte-oriented framing** reserves special byte values for delimiters. If a delimiter appears inside payload, the sender inserts an escape byte; the receiver removes it. This is **byte stuffing**. **Bit-oriented framing** can reserve a bit pattern and insert a zero after a specified run of ones so the reserved flag cannot appear accidentally; the receiver reverses the process.

```text
BYTE_STUFF(payload, FLAG, ESCAPE)
    output <- FLAG
    for each byte b in payload
        if b equals FLAG or ESCAPE
            append ESCAPE
        append b
    append FLAG
    return output
```

Length fields provide another boundary mechanism, but corruption of the length can make a receiver lose synchronization. Practical protocols combine lengths, known headers, delimiters, coding rules, and checks to make resynchronization reliable.

#### **Parity, Checksums, and Cyclic Redundancy Checks**

Error detection adds redundant information calculated from the protected bits.

- A **parity bit** makes the count of one bits even or odd. It detects every odd number of bit flips but misses any even number.
- A **checksum** divides data into words and combines them with arithmetic. Internet checksums are inexpensive in software but have weaker burst-error properties than a well-chosen CRC.
- A **cyclic redundancy check (CRC)** treats a bit string as a polynomial over $GF(2)$ and transmits the remainder of polynomial division by a generator polynomial.

CRC arithmetic uses XOR instead of subtraction because coefficients are 0 or 1. If data polynomial $M(x)$ is shifted by $r$ places for an $r$-degree generator $G(x)$, the sender chooses remainder $R(x)$ so that

$$
T(x)=x^rM(x)+R(x)
$$

is divisible by $G(x)$. The receiver divides the received codeword by the same generator. A non-zero remainder proves that the frame is inconsistent. A zero remainder makes corruption unlikely according to the generator's detection guarantees, but it is not a cryptographic proof and does not stop an attacker who can deliberately recompute the CRC.

#### **Error Detection vs Error Correction**

Detection answers "should this unit be trusted?" Correction answers "can the original information be recovered without retransmission?" Forward error correction adds enough structured redundancy to repair a bounded class of errors. Automatic Repeat reQuest (ARQ) detects failure and asks for another transmission.

| Strategy | Extra cost | Best fit |
|---|---|---|
| Detect and discard | Check bits only | Reliable medium or recovery at a higher layer |
| Detect and retransmit | Check bits, acknowledgments, time, duplicate handling | Errors are occasional and a return path exists |
| Forward error correction | More transmitted redundancy and decoding work | Long delay, broadcast, real-time, or lossy links |
| Hybrid ARQ | FEC first, retransmit when needed | Wireless and cellular links balancing latency and capacity |

#### **Reliable Link Protocols**

A reliable link protocol typically needs sequence numbers, acknowledgments, retransmission rules, and timers. Stop-and-wait sends one frame and waits for an acknowledgment. A sliding window allows several frames in flight, which is essential when link bandwidth-delay product is large.

Link reliability is useful when local recovery is much faster than waiting for end-to-end transport. Wi-Fi, for example, acknowledges unicast frames and can retry locally after radio loss. But local retry has a limit: persistent interference can waste airtime and delay end-to-end recovery. Reliability at one link also cannot guarantee that a later router, host, or application stores the intended data correctly.


In [1]:
def crc_remainder(data_bits: str, generator_bits: str) -> str:
    """Return the CRC remainder using polynomial long division over GF(2)."""

    if generator_bits[0] != "1" or generator_bits[-1] != "1":
        raise ValueError("A generator must begin and end with 1")
    if set(data_bits + generator_bits) - {"0", "1"}:
        raise ValueError("Inputs must be bit strings")

    degree = len(generator_bits) - 1
    working = [int(bit) for bit in data_bits + "0" * degree]
    generator = [int(bit) for bit in generator_bits]

    # Whenever the current leading coefficient is 1, XOR the generator
    # into that position. XOR is subtraction in the two-element field.
    for start in range(len(data_bits)):
        if working[start] == 1:
            for offset, generator_bit in enumerate(generator):
                working[start + offset] ^= generator_bit

    return "".join(str(bit) for bit in working[-degree:])


def crc_is_valid(codeword: str, generator_bits: str) -> bool:
    """Check a received codeword without appending additional zero bits."""

    working = [int(bit) for bit in codeword]
    generator = [int(bit) for bit in generator_bits]
    for start in range(len(codeword) - len(generator_bits) + 1):
        if working[start] == 1:
            for offset, generator_bit in enumerate(generator):
                working[start + offset] ^= generator_bit
    return not any(working[-(len(generator_bits) - 1):])


data = "1101011011"
generator = "10011"
remainder = crc_remainder(data, generator)
codeword = data + remainder

# Flip one bit to model corruption during transmission.
corrupted = list(codeword)
corrupted[5] = "1" if corrupted[5] == "0" else "0"
corrupted = "".join(corrupted)

print("data:       ", data)
print("generator:  ", generator)
print("remainder:  ", remainder)
print("codeword OK:", crc_is_valid(codeword, generator))
print("after flip: ", crc_is_valid(corrupted, generator))


data:        1101011011
generator:   10011
remainder:   1110
codeword OK: True
after flip:  False


### **Multiple Access Protocols**

#### **Channel Partitioning**

When several stations share a medium, a protocol must determine who can transmit. **Channel partitioning** assigns non-overlapping resources:

- TDMA assigns recurring time slots.
- FDMA or OFDMA assigns frequency resources.
- CDMA separates transmissions with codes.
- A scheduled system can allocate time/frequency/spatial resources dynamically.

Partitioning gives predictable isolation when demand is steady, but fixed allocations waste capacity when a station is idle. Dynamic schedulers improve utilization but need coordination, state, and control messages.

#### **ALOHA and Slotted ALOHA**

Random-access protocols let stations transmit without a central schedule and recover after collision. In **pure ALOHA**, a station transmits whenever it has a frame. Any overlapping frame interval causes collision. If aggregate attempts follow the standard Poisson model with offered load $G$ attempts per frame time, normalized throughput is

$$
S_{pure}=Ge^{-2G},
$$

with maximum $1/(2e)\approx 0.184$. **Slotted ALOHA** permits starts only at slot boundaries, halving the vulnerable interval:

$$
S_{slotted}=Ge^{-G},
$$

with maximum $1/e\approx 0.368$ at $G=1$. These are model results, not universal performance promises. They reveal why completely uncoordinated retransmission wastes capacity as load rises.

#### **CSMA, CSMA/CD, and CSMA/CA**

**Carrier Sense Multiple Access (CSMA)** listens before transmitting. If the channel appears busy, a station waits. Carrier sensing reduces collisions, but propagation delay means two distant stations can both observe silence and begin before either signal reaches the other.

Classic shared, half-duplex Ethernet used **CSMA/CD**: listen, transmit, detect a collision while transmitting, send a jam signal, stop, and wait for a randomized binary exponential backoff. Collision detection depends on a station still transmitting when a worst-case collision returns, which connects Ethernet's minimum frame size to network diameter and propagation time. Modern switched full-duplex Ethernet has no shared collision domain, so CSMA/CD is not used on those links.

Wi-Fi generally cannot detect a collision reliably while transmitting because its own radio signal overwhelms a much weaker incoming signal. IEEE 802.11 therefore uses **CSMA/CA**:

```text
CSMA_CA_SEND(frame)
    wait until channel is idle for the required inter-frame interval
    choose random backoff counter from the current contention window
    while counter > 0
        if channel becomes busy
            freeze counter until the channel is idle again
        else
            decrement counter after one idle slot
    transmit frame
    if acknowledgment arrives
        reset contention window
    else
        enlarge contention window and retry up to a limit
```

Optional RTS/CTS exchange reserves airtime before a long data frame. Other stations that hear the reservation update a virtual carrier-sense timer called the Network Allocation Vector. RTS/CTS adds overhead, so it is not automatically beneficial for every frame.

![CSMA/CA flow with carrier sensing, randomized backoff, optional RTS/CTS, data, and acknowledgment.](assets/csma-ca.svg){fig-alt="CSMA CA algorithm with channel sensing, backoff, RTS CTS, data, and acknowledgment" width="58%"}

*Figure source: [Jjgarcia.tsc, CSMA CA, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Csma_ca.svg), reusable with attribution.*

#### **Collision Detection, Avoidance, and Fairness**

In the **hidden terminal** problem, A and C can both reach access point B but cannot hear each other. Each may sense an idle channel and transmit to B simultaneously. Carrier sensing at the sender is therefore incomplete evidence about conditions at the receiver.

![Hidden terminal topology in which two stations cannot hear one another but can both reach the middle receiver.](assets/csma-hidden.svg){fig-alt="Two hidden wireless stations both communicate with a central receiver" width="62%"}

*Figure source: [Mik81, CsmaHidden, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:CsmaHidden.svg), public domain.*

The **exposed terminal** problem is the opposite: a station hears a nearby transmission and waits even though its own transmission to another receiver would not interfere. Avoidance is conservative because a sender cannot perfectly know every receiver's interference conditions.

Random backoff creates probabilistic sharing, not guaranteed fairness. A station with better signal quality, shorter frames, more aggressive parameters, or repeated luck may capture disproportionate airtime. Fairness should also be measured in the right unit. Equal frame counts, equal bytes, equal throughput, and equal airtime are different objectives.


In [2]:
import math
import random


def simulate_slotted_aloha(
    stations: int,
    transmit_probability: float,
    slots: int = 200_000,
    seed: int = 7,
) -> tuple[float, float, float]:
    """Return offered load, throughput, and collision-slot fraction."""

    rng = random.Random(seed)
    attempts = 0
    successes = 0
    collisions = 0

    for _ in range(slots):
        contenders = sum(
            rng.random() < transmit_probability
            for _ in range(stations)
        )
        attempts += contenders
        if contenders == 1:
            successes += 1
        elif contenders > 1:
            collisions += 1

    offered_load = attempts / slots
    throughput = successes / slots
    collision_fraction = collisions / slots
    return offered_load, throughput, collision_fraction


print(" G(sim)  S(sim)  G*exp(-G)  collision slots")
for probability in (0.01, 0.025, 0.05, 0.10):
    offered, throughput, collisions = simulate_slotted_aloha(
        stations=20,
        transmit_probability=probability,
    )
    theoretical = offered * math.exp(-offered)
    print(
        f" {offered:5.3f}   {throughput:5.3f}"
        f"      {theoretical:5.3f}          {collisions:5.3f}"
    )


 G(sim)  S(sim)  G*exp(-G)  collision slots


 0.201   0.167      0.165          0.017


 0.502   0.310      0.304          0.088


 0.999   0.378      0.368          0.263


 2.000   0.269      0.271          0.609


### **Ethernet and MAC Addressing**

#### **Ethernet Frame Structure**

Ethernet is a family of IEEE 802.3 link and physical technologies sharing a common MAC service and frame format across many rates and media. An Ethernet II frame contains:

![Ethernet Type II frame fields and their sizes.](assets/ethernet-type-ii-frame.svg){fig-alt="Ethernet frame with preamble, addresses, EtherType, payload, and frame check sequence" width="94%"}

*Figure source: [Ethernet Type II Frame format, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Ethernet_Type_II_Frame_format.svg), public domain.*

- **Preamble and Start Frame Delimiter** help receiver synchronization and identify the frame start.
- **Destination and source MAC addresses** identify local delivery endpoints.
- **EtherType** identifies the payload protocol, such as IPv4, ARP, or IPv6.
- **Payload** normally carries a higher-layer packet and is padded if needed to meet the minimum frame size.
- **Frame Check Sequence (FCS)** carries a 32-bit CRC checked by the receiver.

The frame length commonly discussed by software excludes some physical overhead, such as preamble and inter-packet gap. When calculating actual link occupancy, state exactly which bytes and intervals are included. A 1,500-byte IP packet does not consume only 1,500 bytes of physical capacity.

#### **MAC Addresses and Broadcast Delivery**

A conventional Ethernet MAC address is 48 bits, written as six hexadecimal octets such as `3c:52:82:1a:7f:09`. Important forms include:

- **Unicast** identifies one interface within the LAN's forwarding context.
- **Multicast** identifies a group; the low-order bit of the first transmitted octet distinguishes group addressing.
- **Broadcast** is `ff:ff:ff:ff:ff:ff` and is delivered throughout the broadcast domain.
- A universally administered address is assigned from an organizational prefix; a locally administered address is deliberately set by software or an administrator.

MAC addresses are not globally routed locators. A switch learns that a source MAC is reachable through a port, but an Internet router does not build a world-wide table of every NIC. Hierarchical IP prefixes provide scalable inter-network forwarding; MAC addresses solve local delivery.

Address randomization also means "MAC address equals permanent device identity" is a poor assumption. Modern clients may use locally administered addresses to reduce passive tracking, especially during wireless scanning or per-network association.

#### **Evolution from Shared Ethernet to Switched Ethernet**

Early Ethernet used shared coaxial cable; later hubs repeated every signal to every port, leaving one collision domain. Bridges and switches learned which MAC addresses were reachable through each port and forwarded selectively. Modern host links are usually switched and full duplex: send and receive can occur concurrently, and each port is its own collision domain.

This evolution retained the frame abstraction while changing access behavior. Textbooks still teach CSMA/CD because it explains Ethernet's history, minimum frame logic, and shared-medium design. It should not be used to explain collisions on a normal full-duplex switch port; duplex mismatch or physical errors are different problems.


In [3]:
from dataclasses import dataclass


@dataclass(frozen=True)
class EthernetHeader:
    destination: str
    source: str
    ether_type: int


def format_mac(raw: bytes) -> str:
    return ":".join(f"{octet:02x}" for octet in raw)


def parse_ethernet_header(frame: bytes) -> EthernetHeader:
    """Parse the 14-byte Ethernet II header visible to software.

    The physical preamble and FCS are often handled by the NIC and may not be
    present in an operating-system packet capture.
    """

    if len(frame) < 14:
        raise ValueError("Frame is shorter than an Ethernet II header")

    return EthernetHeader(
        destination=format_mac(frame[0:6]),
        source=format_mac(frame[6:12]),
        ether_type=int.from_bytes(frame[12:14], byteorder="big"),
    )


# A synthetic broadcast ARP frame: destination, source, EtherType, then payload.
frame = bytes.fromhex(
    "ff ff ff ff ff ff "
    "02 00 00 00 00 24 "
    "08 06 "
    "00 01 08 00 06 04 00 01 "
    "02 00 00 00 00 24 c0 00 02 18 "
    "00 00 00 00 00 00 c0 00 02 01"
)

header = parse_ethernet_header(frame)
print("destination:", header.destination)
print("source:     ", header.source)
print(f"EtherType:   0x{header.ether_type:04x}")
print("payload bytes:", len(frame) - 14)


destination: ff:ff:ff:ff:ff:ff
source:      02:00:00:00:00:24
EtherType:   0x0806
payload bytes: 28


### **Learning Switches and Switched LANs**

#### **Learning, Filtering, Flooding, and Forwarding**

An Ethernet switch maintains a **forwarding database (FDB)** mapping `(VLAN, MAC address)` to an output port and age. It learns from source addresses because the arrival port is evidence that the source is reachable there.

```text
PROCESS_FRAME(frame, incoming_port, vlan)
    learn (vlan, frame.source) -> incoming_port

    if frame.destination is broadcast or relevant multicast
        flood to eligible vlan ports except incoming_port
    else if (vlan, frame.destination) is unknown
        flood to eligible vlan ports except incoming_port
    else if learned output equals incoming_port
        filter: do not send the frame back
    else
        forward only to the learned output port
```

![Three-step switch-learning example: unknown destination flooding, learning from a reply, and later directed forwarding.](assets/switch-learning-flow.svg){fig-alt="A switch learns host A, floods an unknown B destination, learns B from the reply, then forwards directly" width="98%"}

Flooding is not the normal fate of every unicast frame. It is the fallback for unknown destinations and certain group traffic. Dynamic entries expire so that hosts can move, links can change, and stale state eventually disappears.

In software, an exact-match hash table gives average $O(1)$ learning and lookup. High-speed switches use specialized lookup structures and parallel pipelines so a decision can be made at line rate. Capacity still matters: an FDB has finite entries, and excessive source-address churn can cause eviction, flooding, or security controls to activate.

#### **Collision Domains and Broadcast Domains**

Each full-duplex switched port is a separate **collision domain**. Ordinary simultaneous transmissions on different ports do not collide. A **broadcast domain** is the set of ports to which a link-layer broadcast can be propagated. Routers and VLAN boundaries separate broadcast domains; a basic switch extends one.

Separating the terms prevents a common error: adding a switch removes shared Ethernet collisions but does not stop ARP broadcasts from reaching other ports in the same VLAN. Adding more switches can enlarge the broadcast domain unless VLAN or routing design changes.

#### **Loops and the Spanning Tree Protocol**

Redundant switch links improve resilience, but an active layer-2 loop is dangerous. Ethernet frames have no hop limit. A broadcast or unknown unicast can circulate, be copied onto several links, and return indefinitely. Consequences include broadcast storms, duplicate delivery, saturated links, and **MAC flapping** as switches repeatedly learn the same source on different ports.

![Unknown-unicast flooding in a looped layer-2 topology, the problem spanning tree must prevent.](assets/stp-loop-problem.svg){fig-alt="An Ethernet topology loop repeatedly floods an unknown unicast frame" width="55%"}

*Figure source: [Luca Ghio, STP loop problem, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:STP_loop_problem.svg), licensed under CC BY-SA 4.0.*

The **Spanning Tree Protocol (STP)** lets switches exchange Bridge Protocol Data Units and select a loop-free logical tree:

1. Elect the bridge with the lowest bridge identifier as root.
2. Each non-root bridge selects its lowest-cost path toward the root as a root port.
3. Each LAN segment selects a designated port providing the best path toward the root.
4. Other redundant ports do not forward ordinary data frames.

If topology changes, the tree is recomputed and a previously blocked path can become active. Rapid Spanning Tree shortens convergence compared with original STP. The trade-off is that some physical capacity remains inactive for a given tree and traffic may follow a path chosen for loop freedom rather than shortest latency.

#### **VLANs and IEEE 802.1Q**

A **Virtual LAN (VLAN)** creates separate logical broadcast domains on shared switching hardware. An access port commonly associates untagged host traffic with one VLAN. A trunk carries frames for several VLANs and uses an IEEE 802.1Q tag to identify them.

![Ethernet frame before and after insertion of an IEEE 802.1Q VLAN tag.](assets/vlan-tag-insert.svg){fig-alt="An 802.1Q tag inserted between Ethernet source address and EtherType fields" width="98%"}

*Figure source: [Bill Stafford, Ethernet 802.1Q Insert, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Ethernet_802.1Q_Insert.svg), licensed under CC BY-SA 3.0.*

The four-byte tag includes a Tag Protocol Identifier and Tag Control Information containing priority, drop eligibility, and a 12-bit VLAN identifier. A VLAN is not encryption or a complete security boundary. It limits default layer-2 propagation, but routing, trunk configuration, switch control planes, and endpoint policy determine whether traffic can cross boundaries safely.

#### **Link Aggregation and Redundancy**

Link aggregation combines several physical links into one logical bundle for capacity and resilience. Frames from one flow are normally kept on one member based on a hash of header fields so they do not reorder simply because parallel links have different queueing delay. One very large flow may therefore remain limited to one member even when the aggregate has unused capacity.

Aggregation handles parallel links between the participating systems; it does not replace spanning tree or routing for arbitrary topologies. Designs must distinguish fast local member failure, switch failure, control-plane failure, and complete path failure.


In [4]:
from dataclasses import dataclass


@dataclass
class LearnedLocation:
    port: int
    last_seen: float


class LearningSwitch:
    """Small educational model of one VLAN on an Ethernet switch."""

    BROADCAST = "ff:ff:ff:ff:ff:ff"

    def __init__(self, ports: set[int], aging_seconds: float = 300.0):
        self.ports = ports
        self.aging_seconds = aging_seconds
        self.fdb: dict[str, LearnedLocation] = {}

    def _expire(self, now: float) -> None:
        stale = [
            mac
            for mac, location in self.fdb.items()
            if now - location.last_seen > self.aging_seconds
        ]
        for mac in stale:
            del self.fdb[mac]

    def process(
        self,
        source: str,
        destination: str,
        incoming_port: int,
        now: float,
    ) -> tuple[str, list[int]]:
        self._expire(now)

        # Learning always uses the source address and arrival port.
        self.fdb[source] = LearnedLocation(incoming_port, now)

        if destination == self.BROADCAST or destination not in self.fdb:
            outputs = sorted(self.ports - {incoming_port})
            return "flood", outputs

        output = self.fdb[destination].port
        if output == incoming_port:
            return "filter", []
        return "forward", [output]


switch = LearningSwitch(ports={1, 2, 3}, aging_seconds=30)
frames = [
    ("A", "B", 1, 0),   # B is unknown, so flood.
    ("B", "A", 2, 1),   # Learn B; A is known on port 1.
    ("A", "B", 1, 2),   # Both locations are now known.
    ("C", "A", 3, 40),  # A and B aged out; A becomes unknown again.
]

for source, destination, port, now in frames:
    action, outputs = switch.process(source, destination, port, now)
    print(
        f"t={now:>2}  {source}->{destination} on P{port}: "
        f"{action:<7} {outputs}"
    )


t= 0  A->B on P1: flood   [2, 3]
t= 1  B->A on P2: forward [1]
t= 2  A->B on P1: forward [2]
t=40  C->A on P3: flood   [1, 2]


### **Local Resolution and Host Attachment**

#### **Address Resolution Protocol**

After IP routing chooses a next-hop IPv4 address on an Ethernet-like link, the sender needs the corresponding MAC address. The **Address Resolution Protocol (ARP)**, specified by [RFC 826](https://datatracker.ietf.org/doc/html/rfc826), distributes this local mapping dynamically.

For a host asking for `192.0.2.1`:

1. It checks its ARP cache.
2. On a miss, it broadcasts an ARP request: "Who has 192.0.2.1? Tell 192.0.2.24."
3. Every station in the broadcast domain receives the request, but the target normally answers.
4. The ARP reply states the target's MAC address and is usually unicast to the requester.
5. The requester caches the mapping and can construct the data frame.

![ARP request broadcast and ARP reply between hosts on a local network.](assets/arp-query.svg){fig-alt="One host broadcasts an ARP query and the target returns its hardware address" width="62%"}

*Figure source: [Papapep, Protocol ARP, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Protocol_ARP.svg), licensed under CC BY-SA 4.0.*

ARP resolves the **next hop**, not necessarily the final IP destination. If the destination is on the sender's subnet, the next hop is the destination itself. If it is remote, the next hop is the default gateway or another router selected by the route table.

#### **ARP Caches, Gratuitous ARP, and Failure Modes**

ARP cache entries are soft state and expire. Caching avoids a broadcast before every packet but creates possible staleness after an address moves or a device is replaced. Operating systems use states and timers rather than treating every mapping as permanently valid.

A **gratuitous ARP** announces or probes an address without waiting for a conventional request. It can update peers after failover, detect duplicate addressing, or announce a moved service. Because classic ARP has no authentication, a malicious or misconfigured host can advertise a false mapping. ARP spoofing can redirect, intercept, or disrupt local traffic; switch security features and higher-layer authentication reduce but do not eliminate the risk.

Common failure patterns include duplicate IP addresses, stale entries, hosts in different VLANs, blocked broadcasts, an incorrect subnet mask, or a gateway interface that is down. Clearing a cache may temporarily help staleness, but it does not fix the underlying topology or configuration error.

IPv6 does not use ARP. **Neighbor Discovery Protocol (NDP)** uses ICMPv6 and solicited-node multicast for address resolution, router discovery, reachability, prefix information, and duplicate-address detection. Treating NDP as "ARP with larger addresses" hides these additional responsibilities and its different security considerations.

#### **First-Hop Delivery to a Default Gateway**

The host first performs an IP prefix decision. If destination $D$ and local address $A$ share the configured prefix of length $p$, delivery is on-link. Otherwise, the route table selects a gateway. In conceptual bit terms:

$$
(A \land mask_p) = (D \land mask_p)
$$

means the addresses share that prefix. Real route selection uses longest-prefix matching across all routes, not only a single subnet mask and default route. Chapter 3 develops that process fully.

![A wireless host resolves the default gateway MAC, bridges a frame through the LAN, and leaves the link at the router.](assets/first-hop-delivery.svg){fig-alt="Laptop sends a frame through Wi-Fi and a VLAN switch to its default gateway while keeping the remote server IP destination" width="98%"}

The crucial address combination for a remote destination is:

| Field | Value on the first local frame |
|---|---|
| Ethernet/Wi-Fi source | Host's local link address |
| Ethernet/Wi-Fi destination | Default gateway's link address |
| IP source | Host's IP address, before any NAT |
| IP destination | Remote server's IP address |

The access point and switches bridge the frame without becoming its IP destination. The gateway consumes the local frame because its MAC address matches, then forwards the enclosed IP datagram using a new frame on another link.


In [5]:
from ipaddress import ip_address, ip_interface


def choose_arp_target(
    local_interface: str,
    destination_ip: str,
    default_gateway: str,
) -> str:
    """Choose the IPv4 address whose MAC must be resolved on a simple host.

    This model has one connected prefix and one default route. Real route
    tables can contain many more-specific routes and policy decisions.
    """

    interface = ip_interface(local_interface)
    destination = ip_address(destination_ip)

    if destination in interface.network:
        return str(destination)       # Resolve the destination itself.
    return str(ip_address(default_gateway))  # Resolve the selected router.


local = "192.0.2.24/24"
gateway = "192.0.2.1"

for destination in ("192.0.2.77", "203.0.113.80"):
    target = choose_arp_target(local, destination, gateway)
    scope = "on-link host" if target == destination else "default gateway"
    print(f"destination {destination:<13} -> ARP for {target:<12} ({scope})")


destination 192.0.2.77    -> ARP for 192.0.2.77   (on-link host)
destination 203.0.113.80  -> ARP for 192.0.2.1    (default gateway)


### **Wireless LANs**

#### **IEEE 802.11 Architecture and Association**

An 802.11 **station (STA)** is a wireless interface participating in a WLAN. In infrastructure mode, stations associate with an **access point (AP)**. The AP and its associated stations form a **Basic Service Set (BSS)** identified on the air by a BSSID. Several BSSs can present the same network name (SSID) and connect through a distribution system to form an **Extended Service Set (ESS)**.

![Generic IEEE 802.11 architecture with stations, access points, basic service sets, and a distribution system.](assets/wifi-network-architecture.svg){fig-alt="Wireless stations associate with access points connected through an 802.11 distribution system" width="76%"}

*Figure source: [Superspritz, 802.11 Network Architecture, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:802.11_Network_Architecture.svg), licensed under CC BY-SA 4.0.*

Joining involves several distinct stages:

1. **Discovery:** listen for beacons or send probe requests to find candidate BSSs.
2. **Authentication and association:** establish 802.11 state with a selected AP.
3. **Link security:** when protected access is used, authenticate credentials and derive encryption/integrity keys.
4. **Network configuration:** obtain IP addressing, routes, and DNS information.
5. **Connectivity validation:** the operating system may test whether Internet access or a captive portal is present.

Seeing an SSID only proves that a beacon or probe response was received. It does not prove successful association, key establishment, DHCP, DNS, routing, or Internet access. Troubleshooting should identify the failed stage.

#### **Hidden and Exposed Terminal Problems**

Wireless reception is local and asymmetric. A station's carrier-sense decision may differ from the receiver's interference environment. Hidden terminals motivate randomized backoff, acknowledgments, and sometimes RTS/CTS. Exposed terminals show the cost of conservative deferral. Directional antennas, power differences, multiple channels, and modern multi-user scheduling make real radio coordination more complex than one shared circle.

#### **Acknowledgments, Retransmissions, and Rate Adaptation**

Unicast 802.11 data is normally followed by a link-layer acknowledgment after a short inter-frame space. Missing acknowledgment suggests that the data or ACK was not decoded, so the sender may retry. The retry bit and sequence information help a receiver recognize duplicates.

Rate adaptation selects a modulation and coding scheme according to observed conditions. A high nominal PHY rate is valuable only when frames decode reliably. Using an aggressive rate can cause retries; using a very conservative rate consumes more airtime and slows every station sharing the channel.

For a simplified exchange,

$$
T_{air} \approx T_{contention}+T_{preamble}+\frac{L_{frame}}{R_{PHY}}+T_{SIFS}+T_{ACK}.
$$

Useful goodput is application payload divided by the total exchange time, including headers, contention, acknowledgments, and retries. This is why a "1.2 Gb/s" Wi-Fi association does not imply 1.2 Gb/s application transfer.

#### **Roaming and Mobility Within a WLAN**

When several APs advertise one SSID, the client generally decides when to roam based on signal, quality, policy, and implementation heuristics. Association moves to another BSS; the distribution network must update where the client's MAC is reachable. Security key caching and fast-transition mechanisms can reduce interruption.

Layer-2 roaming does not automatically solve IP mobility. If the new AP bridges into the same IP subnet, the client may keep its address. Crossing into another routed subnet can require new configuration and can break transport connections unless another mobility mechanism preserves continuity. The same SSID is a user-facing name, not proof of one broadcast domain or one routing design.


In [6]:
def wifi_exchange(
    payload_bytes: int,
    phy_rate_mbps: float,
    frame_overhead_bytes: int = 38,
    preamble_us: float = 40,
    contention_us: float = 80,
    sifs_us: float = 16,
    ack_us: float = 44,
    frame_error_probability: float = 0.0,
) -> tuple[float, float]:
    """Estimate expected airtime and payload goodput for one Wi-Fi frame.

    Values are illustrative rather than tied to one exact 802.11 PHY. With
    independent frame failures, expected attempts equal 1 / (1 - p).
    """

    if not 0 <= frame_error_probability < 1:
        raise ValueError("Frame error probability must be in [0, 1)")

    transmitted_bits = (payload_bytes + frame_overhead_bytes) * 8
    data_us = transmitted_bits / phy_rate_mbps
    one_attempt_us = contention_us + preamble_us + data_us + sifs_us + ack_us
    expected_attempts = 1 / (1 - frame_error_probability)
    expected_airtime_us = one_attempt_us * expected_attempts

    goodput_mbps = (payload_bytes * 8) / expected_airtime_us
    return expected_airtime_us, goodput_mbps


print("PHY rate  error p  expected airtime  payload goodput")
for rate, error_probability in ((54, 0.02), (300, 0.02), (300, 0.20)):
    airtime, goodput = wifi_exchange(
        payload_bytes=1_500,
        phy_rate_mbps=rate,
        frame_error_probability=error_probability,
    )
    print(
        f"{rate:>7} Mb/s   {error_probability:>4.0%}"
        f"       {airtime:7.1f} us       {goodput:7.1f} Mb/s"
    )


PHY rate  error p  expected airtime  payload goodput
     54 Mb/s     2%         416.2 us          28.8 Mb/s
    300 Mb/s     2%         225.5 us          53.2 Mb/s
    300 Mb/s    20%         276.3 us          43.4 Mb/s


### **Access Network Technologies**

#### **Residential Broadband and Fiber Access**

The access network is often the narrowest and most variable part of an Internet path.

- **DSL** uses telephone copper pairs, with achievable rate depending strongly on line length and quality.
- **Cable access** uses a hybrid fiber/coax plant. Capacity is shared within service groups and scheduled by the access system.
- **Passive Optical Network (PON)** shares optical distribution using passive splitters. Downstream traffic is broadcast optically to an appropriate set of endpoints and upstream transmission is scheduled to avoid collisions.
- **Point-to-point fiber** gives each subscriber a dedicated optical link to an aggregation device, though upstream capacity is still shared later.
- **Fixed wireless access** uses a radio link from premises equipment to a provider base station.

"Fiber" describes a medium, not an automatic end-to-end guarantee. The customer plan, optical split, provider aggregation, backhaul, peering, home router, Wi-Fi, and server path all influence measured performance. Access plans may also be asymmetric because many residential workloads historically receive more than they send.

#### **Enterprise and Campus Access Networks**

Campus designs commonly use access switches and APs connected to distribution or core layers. VLANs separate groups; routing controls communication between them; 802.1X can authenticate a user or device before granting access; Power over Ethernet can supply APs, phones, and cameras through twisted pair.

Operational scale changes the design. Redundant uplinks, link aggregation, spanning-tree or routed-access choices, centralized WLAN control, multicast handling, quality of service, and telemetry become important. A campus LAN is not merely a larger home switch: it has administrative policy, failure domains, and capacity planning.

#### **Cellular and Satellite Access Overview**

In cellular access, user equipment communicates over scheduled radio resources with a base station, then traffic crosses a provider transport and packet core before reaching the public Internet or a private service. Mobility, authentication, bearer/session state, radio scheduling, and tunnelling are integral. The phone may expose an IP interface to applications even though its underlying radio frames are not Ethernet.

Satellite access adds a space segment. A geostationary satellite's large distance creates substantial propagation delay; low-Earth-orbit systems reduce that distance but require moving constellations, handovers, and ground infrastructure. Queueing and capacity sharing still apply in either case.

| Access type | Shared resource | Characteristic concern |
|---|---|---|
| Switched Ethernet | Uplinks and upstream aggregation | Cabling rate, duplex, queueing, oversubscription |
| Wi-Fi | Channel airtime | Interference, contention, signal quality, retries |
| Cable/PON | Local access segment and aggregation | Scheduling, service-group load, plan shaping |
| Cellular | Scheduled radio and provider core | Coverage, load, mobility, variable rate |
| Satellite | Radio spectrum and space/ground path | Propagation, weather, handover, shared capacity |

The correct comparison is therefore not only advertised peak rate. Evaluate expected goodput, latency distribution, loss/retry behavior, coverage, mobility, contention scope, upload capacity, and failure recovery.


### **Following a Packet Across the First Hop**

Suppose the laptop has `192.0.2.24/24`, default gateway `192.0.2.1`, and has resolved the website to `203.0.113.80`.

1. **Route decision:** `203.0.113.80` is not in the connected `/24`, so the selected next hop is `192.0.2.1`.
2. **Neighbor resolution:** if no cache entry exists, the laptop sends an ARP request through its Wi-Fi BSS. The AP bridges the broadcast into the appropriate VLAN.
3. **Switch propagation:** switches flood the request within VLAN 20 and learn the laptop's source MAC on the incoming path.
4. **Gateway reply:** the router interface owning `192.0.2.1` returns its MAC. Switches learn the gateway source location while forwarding the reply toward the laptop.
5. **Data encapsulation:** the laptop creates an IP datagram whose destination is `203.0.113.80`, then places it in an 802.11 data frame whose local receiver path leads to the gateway MAC.
6. **Medium access:** the laptop senses the channel, performs randomized backoff, transmits, and expects a link acknowledgment.
7. **Bridging:** the AP converts the wireless delivery into the distribution-system representation; switches forward it through VLAN 20 according to their FDBs.
8. **Routing boundary:** the gateway removes the local frame, processes the IP datagram, decrements TTL, selects an IP next hop, and creates different link-layer framing.

Several pieces of state cooperate:

| State | Owner | Key question answered |
|---|---|---|
| Association and security keys | Station and AP | May this station exchange protected wireless frames? |
| Switch FDB | Each switch | Which port reaches this `(VLAN, source MAC)`? |
| ARP cache | Host and router | Which local MAC corresponds to this next-hop IPv4 address? |
| Route table | Host and router | Which next hop and interface should carry this IP destination? |
| Queue/backoff state | NIC, AP, switch | When can this frame use the constrained resource? |

None of those tables is a complete end-to-end route. They solve different local decisions at different layers.


### **Observing and Troubleshooting the Link Layer**

Windows exposes useful local evidence without requiring a custom program:

```powershell
# Interface descriptions, physical addresses, status, and speed
Get-NetAdapter
getmac /v

# IPv4 ARP and IPv6 neighbor-cache state
arp -a
Get-NetNeighbor

# Current Wi-Fi association, channel, signal, and negotiated rates
netsh wlan show interfaces

# IP prefixes and gateways that determine the next hop
Get-NetIPConfiguration
Get-NetRoute
```

Wireshark filters connect the chapter to actual frames:

```text
eth                  # Ethernet frames
arp                  # ARP requests and replies
eth.addr == aa:bb:cc:dd:ee:ff
vlan                 # Captured 802.1Q tags
wlan                 # 802.11 frames, when capture hardware exposes them
wlan.fc.retry == 1   # Wireless retransmission flag
```

Captures made on a normal laptop often occur after NIC processing. The preamble and FCS may be absent; VLAN tags may be inserted or removed by hardware offload; Wi-Fi monitor-mode metadata may be unavailable. A missing field in a capture does not prove that it never existed on the wire.

Use symptoms to choose evidence:

| Symptom | Likely link/access cause | Evidence |
|---|---|---|
| Interface has no carrier or cannot associate | Cable, optics, radio, authentication | Adapter status, AP logs, signal, link LEDs/counters |
| IP is configured but gateway is unreachable | VLAN, ARP/NDP, wrong prefix, port policy | ARP/neighbor state, switch VLAN/FDB, packet capture |
| Intermittent high latency on Wi-Fi | Contention, interference, weak signal, retries | Channel utilization, retry rate, signal, wired comparison |
| Only one VLAN fails | Access/trunk mismatch or routing boundary | Port mode, allowed VLANs, tag capture, gateway interface |
| Broadcast storm and unstable MAC locations | Layer-2 loop | STP state, topology, MAC move logs, interface rates |
| Link rate is high but transfer is slow | Shared airtime/uplink, errors, duplex, shaping, higher layers | Error counters, goodput, retries, queues, path comparison |

The most useful control test is often a comparison: Wi-Fi versus wired, one AP versus another, same VLAN versus routed destination, or local gateway versus remote server. A controlled difference narrows the failed mechanism.


### **Comparison and Summary**

The chapter's key distinctions are:

| Do not confuse | Correct distinction |
|---|---|
| Frame and IP datagram | One-link container versus internetwork packet |
| MAC address and IP address | Local forwarding identifier versus hierarchical routed locator |
| Collision domain and broadcast domain | Shared collision scope versus layer-2 flooding scope |
| Switch learning and ARP | MAC-to-port state inside a switch versus IPv4-to-MAC state at a node |
| Error detection and reliability | Recognizing corruption versus recovering accepted data |
| PHY rate and goodput | Symbol/data transmission rate versus useful delivered payload rate |
| CSMA/CD and CSMA/CA | Detecting shared-wire collisions versus avoiding radio contention and inferring success by ACK |
| VLAN and security boundary | Logical broadcast separation versus complete authorization/confidentiality policy |
| Same SSID and same subnet | Wireless network name versus IP broadcast/routing design |

A first-hop delivery can now be explained as one coherent mechanism:

1. Physical and link technologies provide a way to exchange bounded frames with an adjacent node.
2. Framing and CRC let a receiver locate a unit and reject many corrupt units.
3. A multiple-access protocol or scheduler coordinates a shared medium.
4. Ethernet or 802.11 addresses identify local delivery participants.
5. Switches learn source locations and selectively forward within a VLAN.
6. STP or another loop-avoidance design prevents indefinite layer-2 circulation.
7. ARP or IPv6 NDP resolves the selected next hop to a local link address.
8. The default gateway ends the local frame's scope and begins the next IP-forwarding decision.

Chapter 3 starts exactly at that boundary: how IP addresses are structured, how a router performs longest-prefix matching, and how forwarding state moves a datagram across one routed hop after another.
